In [30]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import cv2

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.layers import Input, Concatenate, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import Sequence

import tensorflow.keras.backend as K

In [31]:
metadata_path = r"D:\archive\HAM10000_metadata.csv"

image_dir_1 = r"D:\archive\HAM10000_images_part_1"
image_dir_2 = r"D:\archive\HAM10000_images_part_2"

In [32]:
df = pd.read_csv(metadata_path)

df.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [33]:
image_paths = []

for img_id in df['image_id']:

    path1 = os.path.join(image_dir_1, img_id + ".jpg")
    path2 = os.path.join(image_dir_2, img_id + ".jpg")

    if os.path.exists(path1):
        image_paths.append(path1)
    else:
        image_paths.append(path2)

df['image_path'] = image_paths

In [10]:
df['age'] = df['age'].fillna(df['age'].median())
df['sex'] = df['sex'].fillna("unknown")
df['localization'] = df['localization'].fillna("unknown")
df['dx_type'] = df['dx_type'].fillna("unknown")

In [34]:
label_encoder = LabelEncoder()

df['label'] = label_encoder.fit_transform(df['dx'])

num_classes = df['label'].nunique()

print("Classes:", label_encoder.classes_)

Classes: ['akiec' 'bcc' 'bkl' 'df' 'mel' 'nv' 'vasc']


In [35]:
sex_encoder = LabelEncoder()
df['sex'] = sex_encoder.fit_transform(df['sex'])

loc_encoder = LabelEncoder()
df['localization'] = loc_encoder.fit_transform(df['localization'])

dx_type_encoder = LabelEncoder()
df['dx_type'] = dx_type_encoder.fit_transform(df['dx_type'])

In [36]:
scaler = StandardScaler()

df['age'] = scaler.fit_transform(df[['age']])

In [28]:
!pip install opencv-python

  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/40.2 MB ? eta -:--:--
    --------------------------------------- 0.5/40.2 MB 2.0 MB/s eta 0:00:21
   - -------------------------------------- 1.3/40.2 MB 2.3 MB/s eta 0:00:17
   - -------------------------------------- 1.8/40.2 MB 2.5 MB/s eta 0:00:16
   -- ------------------------------------- 2.6/40.2 MB 2.7 MB/s eta 0:00:14
   --- ------------------------------------ 3.1/40.2 MB 2.8 MB/s eta 0:00:14
   --- ------------------------------------ 3.9/40.2 MB 2.8 MB/s eta 0:00:13
   ---- ----------------------------------- 4.7/40.2 MB 3.0 MB/s eta 0:00:13
   ----- ---------------------------------- 5.5/40.2 MB 3.1 MB/s eta 0:00:12
   ------ --------------------------------- 6.3/40.2 MB 3.1 MB/s eta 0:00:11
   ------- ------

In [37]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

In [38]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(df['label']),
    y=df['label']
)

class_weights = dict(enumerate(class_weights))

In [39]:
def focal_loss(gamma=2., alpha=.25):

    def focal_loss_fixed(y_true, y_pred):

        epsilon = 1e-8
        y_pred = K.clip(y_pred, epsilon, 1. - epsilon)

        cross_entropy = -y_true * K.log(y_pred)

        weight = alpha * K.pow(1 - y_pred, gamma)

        loss = weight * cross_entropy

        return K.sum(loss, axis=1)

    return focal_loss_fixed

In [40]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

for layer in base_model.layers[:-80]:
    layer.trainable = False

x = base_model.output

x = GlobalAveragePooling2D()(x)

x = BatchNormalization()(x)

x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

image_features = Dense(128, activation='relu')(x)

In [41]:
meta_input = Input(shape=(4,))

m = Dense(32, activation='relu')(meta_input)
m = BatchNormalization()(m)

meta_features = Dense(16, activation='relu')(m)

In [42]:
combined = Concatenate()([image_features, meta_features])

z = Dense(128, activation='relu')(combined)

z = Dropout(0.5)(z)

output = Dense(num_classes, activation='softmax')(z)

In [43]:
model = Model(
    inputs=[base_model.input, meta_input],
    outputs=output
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)    │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1 (Conv2D)                │ (None, 112, 112, 32)      │             864 │ input_layer_2[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bn_Conv1 (BatchNormalization) │ (None, 112, 112, 32)      │             128 │ Conv1[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1_relu (ReLU)             │ (None, 112, 112, 32)      │               0 │ bn_Conv1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise       │ (None, 112, 112, 32)      │             288 │ Conv1_relu[0][0]           │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_BN    │ (None, 112, 112, 32)      │             128 │ expanded_conv_depthwise[0… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_relu  │ (None, 112, 112, 32)      │               0 │ expanded_conv_depthwise_B… │
│ (ReLU)                        │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project         │ (None, 112, 112, 16)      │             512 │ expanded_conv_depthwise_r… │
│ (Conv2D)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project_BN      │ (None, 112, 112, 16)      │              64 │ expanded_conv_project[0][… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand (Conv2D)       │ (None, 112, 112, 96)      │           1,536 │ expanded_conv_project_BN[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_BN             │ (None, 112, 112, 96)      │             384 │ block_1_expand[0][0]       │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_relu (ReLU)    │ (None, 112, 112, 96)      │               0 │ block_1_expand_BN[0][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_pad (ZeroPadding2D)   │ (None, 113, 113, 96)      │               0 │ block_1_expand_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_depthwise             │ (None, 56, 56, 96)        │             864 │ block_1_pad[0][0]          │
│ (DepthwiseConv2D)             │                           │               

 Total params: 2,644,215 (10.09 MB)

 Trainable params: 2,451,319 (9.35 MB)

 Non-trainable params: 192,896 (753.50 KB)

In [44]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss=focal_loss(),
    metrics=['accuracy']
)

In [46]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=4
)

In [50]:
class SkinDataGenerator(Sequence):

    def __init__(self, dataframe, batch_size=32):

        super().__init__()   # fixes keras warning

        self.df = dataframe.reset_index(drop=True)
        self.batch_size = batch_size
        self.indexes = np.arange(len(self.df))

    def __len__(self):

        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, index):

        batch_indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]

        batch_df = self.df.iloc[batch_indexes]

        images = []
        meta = []
        labels = []

        for _, row in batch_df.iterrows():

            img = cv2.imread(row['image_path'])
            img = cv2.resize(img,(224,224))
            img = img / 255.0

            images.append(img)

            meta.append([
                row['age'],
                row['sex'],
                row['localization'],
                row['dx_type']
            ])

            labels.append(row['label'])

        images = np.array(images, dtype=np.float32)
        meta = np.array(meta, dtype=np.float32)

        labels = tf.keras.utils.to_categorical(labels, num_classes)

        # IMPORTANT: return tuple instead of list
        return (images, meta), labels

In [51]:
batch_size = 32

train_generator = SkinDataGenerator(train_df, batch_size)

test_generator = SkinDataGenerator(test_df, batch_size)

In [52]:
history = model.fit(

    train_generator,

    validation_data=test_generator,

    epochs=50,

    class_weight=class_weights,

    callbacks=[early_stop, reduce_lr]

)

Epoch 1/50
251/251 ━━━━━━━━━━━━━━━━━━━━ 444s 2s/step - accuracy: 0.2949 - loss: 0.3563 - val_accuracy: 0.6710 - val_loss: 0.2685 - learning_rate: 3.0000e-04
Epoch 2/50
251/251 ━━━━━━━━━━━━━━━━━━━━ 417s 2s/step - accuracy: 0.3969 - loss: 0.2544 - val_accuracy: 0.6640 - val_loss: 0.2787 - learning_rate: 3.0000e-04
Epoch 3/50
251/251 ━━━━━━━━━━━━━━━━━━━━ 533s 2s/step - accuracy: 0.4754 - loss: 0.1973 - val_accuracy: 0.6540 - val_loss: 0.1656 - learning_rate: 3.0000e-04
Epoch 4/50
251/251 ━━━━━━━━━━━━━━━━━━━━ 520s 2s/step - accuracy: 0.5198 - loss: 0.1438 - val_accuracy: 0.6465 - val_loss: 0.2163 - learning_rate: 3.0000e-04
Epoch 5/50
251/251 ━━━━━━━━━━━━━━━━━━━━ 450s 2s/step - accuracy: 0.6038 - loss: 0.1278 - val_accuracy: 0.5502 - val_loss: 0.5059 - learning_rate: 3.0000e-04
Epoch 6/50
251/251 ━━━━━━━━━━━━━━━━━━━━ 385s 2s/step - accuracy: 0.5849 - loss: 0.1243 - val_accuracy: 0.6740 - val_loss: 0.2602 - learning_rate: 3.0000e-04
Epoch 7/50
251/251 ━━━━━━━━━━━━━━━━━━━━ 505s 2s/step - acc

In [54]:
loss, accuracy = model.evaluate(test_generator)

print("Test Accuracy:", accuracy)

63/63 ━━━━━━━━━━━━━━━━━━━━ 45s 710ms/step - accuracy: 0.7504 - loss: 0.1116
Test Accuracy: 0.750374436378479


In [55]:
def predict_image(image_path, age, sex, localization, dx_type):

    img = cv2.imread(image_path)
    img = cv2.resize(img,(224,224))
    img = img/255.0
    img = np.expand_dims(img, axis=0)

    sex = sex_encoder.transform([sex])
    localization = loc_encoder.transform([localization])
    dx_type = dx_type_encoder.transform([dx_type])

    age = scaler.transform([[age]])[0][0]

    meta = np.array([[age, sex[0], localization[0], dx_type[0]]])

    prediction = model.predict([img, meta])

    predicted_class = label_encoder.inverse_transform([np.argmax(prediction)])

    print("Prediction:", predicted_class[0])
    print("Confidence:", np.max(prediction))

In [58]:
predict_image(
    r"D:\archive\HAM10000_images_part_1\ISIC_0024306.jpg",
    age=45,
    sex="male",
    localization="back",
    dx_type="histo"
)

C:\Users\DELL\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Prediction: nv
Confidence: 0.9959448


In [69]:
model.save("skin_disease_model.keras")

In [70]:
import joblib

joblib.dump(label_encoder,"label_encoder.pkl")
joblib.dump(sex_encoder,"sex_encoder.pkl")
joblib.dump(loc_encoder,"loc_encoder.pkl")
joblib.dump(dx_type_encoder,"dx_type_encoder.pkl")
joblib.dump(scaler,"age_scaler.pkl")

['age_scaler.pkl']

In [64]:
import sys
print(sys.version)

3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


In [65]:
from sklearn.metrics import classification_report, accuracy_score

y_true = []
y_pred = []

for (images, meta), labels in test_generator:

    predictions = model.predict((images, meta))

    y_true.extend(np.argmax(labels, axis=1))
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("Accuracy:", accuracy_score(y_true, y_pred))

print("\nClassification Report:\n")

print(classification_report(
    y_true,
    y_pred,
    target_names=label_encoder.classes_
))

1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 825ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 721ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 689ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 720ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 728ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 757ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 791ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 812ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 797ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 835ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 857ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 951ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 859ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 763ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 823ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 943ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 699ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 811ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step   
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 990ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 737ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 749ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 739ms/

In [66]:
import tensorflow as tf
print(tf.__version__)

2.20.0
